# Phase 9b — Replication Analysis (ds004789)

Compares CD-B2M performance across:
- **Phase 8 reference dataset** (from `cdb2m-phase8`)
- **ds004789 replication dataset** (from `cdb2m-phase9a`)

Quantifies cross-dataset generalization, causal advantage, and representation alignment (CKA).


In [1]:
import sys, pathlib, warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
warnings.filterwarnings('ignore')

# ── Output Directory ───────────────────────────────────────────────────
OUT_DIR = pathlib.Path('/kaggle/working/phase9b_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Auto-Detect Phase 8 and Phase 9a Paths ─────────────────────────────
def find_file(patterns, search_roots):
    if isinstance(patterns, str):
        patterns = [patterns]
    for pat in patterns:
        for root in search_roots:
            r_path = pathlib.Path(root)
            if r_path.exists():
                matches = list(r_path.rglob(pat))
                if matches:
                    return matches[0]
    return None

search_dirs = ['/kaggle/input', '/kaggle/working']

# Prefer X_real.npy, then X_region.npy, then any X*.npy
P8_X_PATH = find_file(['X_real.npy', 'X_region.npy', 'X*.npy'], search_dirs)
P8_M_PATH = find_file(['M_real.csv', 'M_region.csv', 'M*.csv'], search_dirs)

P9A_X_PATH = find_file('X_fr1.npy', search_dirs)
P9A_M_PATH = find_file('M_fr1.csv', search_dirs)

print('Phase 8 X found  :', P8_X_PATH)
print('Phase 8 M found  :', P8_M_PATH)
print('Phase 9a X found :', P9A_X_PATH)
print('Phase 9a M found :', P9A_M_PATH)
print('Output dir       :', OUT_DIR)


Phase 8 X found  : /kaggle/input/datasets/rachana240906/cdb2m-phase8/phase8_out/X_region.npy
Phase 8 M found  : /kaggle/input/datasets/rachana240906/cdb2m-phase8/phase8_out/M_region.csv
Phase 9a X found : /kaggle/input/datasets/rachana240906/cdb2m-phase9a/phase9a_out/X_fr1.npy
Phase 9a M found : /kaggle/input/datasets/rachana240906/cdb2m-phase9a/phase9a_out/M_fr1.csv
Output dir       : /kaggle/working/phase9b_out


In [2]:
import importlib.util, sys, pathlib

# Search for cdb2m_lib.py in inputs
lib_candidates = list(pathlib.Path('/kaggle/input').rglob('cdb2m_lib.py')) + [pathlib.Path('/kaggle/working/cdb2m_lib.py')]
lib_path = lib_candidates[0] if lib_candidates else None

loaded_lib = False
if lib_path and lib_path.exists():
    try:
        spec = importlib.util.spec_from_file_location('cdb2m_lib', str(lib_path))
        cdb2m_lib = importlib.util.module_from_spec(spec)
        sys.modules['cdb2m_lib'] = cdb2m_lib
        spec.loader.exec_module(cdb2m_lib)
        from cdb2m_lib import (
            CORE, BANDS, PAD, SEP, MAX_LEN,
            BrainEncoder, SeqGRU, make_folds, make_sequence, to_input, holm
        )
        print('cdb2m_lib loaded successfully from:', lib_path)
        loaded_lib = True
    except Exception as e:
        print('Error importing from cdb2m_lib file:', e)

if not loaded_lib:
    print('Using self-contained constants.')
    CORE = ['hippocampus', 'amygdala', 'parahippocampal', 'lateral_temporal']
    BANDS = ['theta', 'alpha', 'beta', 'lgamma', 'hgamma']
    PAD, SEP, MAX_LEN = 0, 1, 12


cdb2m_lib loaded successfully from: /kaggle/input/datasets/rachana240906/cdb2m-phase8/phase8_out/cdb2m_lib.py


In [3]:
import torch.nn as nn, torch, torch.nn.functional as F

VOCAB_SIZE = 64
D_PROJ     = 64  # Shared latent space dimension for InfoNCE & CKA

class AdaptiveBrainEncoder(nn.Module):
    """
    Adapts 4D Phase 8 (B, 4, 16, 5) or 3D ds004789 (B, 4, 5)
    to a shared projection latent space of dimension D_PROJ.
    """
    def __init__(self, d_proj=D_PROJ):
        super().__init__()
        self.proj_4d = nn.Sequential(
            nn.Linear(4 * 16 * 5, 128),
            nn.GELU(),
            nn.Linear(128, d_proj),
        )
        self.proj_3d = nn.Sequential(
            nn.Linear(4 * 5, 128),
            nn.GELU(),
            nn.Linear(128, d_proj),
        )
    def forward(self, x):
        x_clean = torch.nan_to_num(x, nan=0.0)
        if x_clean.ndim == 4:
            return self.proj_4d(x_clean.flatten(1))
        else:
            return self.proj_3d(x_clean.flatten(1))

class TaskModelGRU(nn.Module):
    """
    Sequence GRU task model with projection head to match brain embedding dim.
    """
    def __init__(self, n_tokens=VOCAB_SIZE, d_emb=32, d_hid=128, d_proj=D_PROJ):
        super().__init__()
        self.emb = nn.Embedding(n_tokens, d_emb, padding_idx=0)
        self.gru = nn.GRU(d_emb, d_hid, batch_first=True)
        self.classifier = nn.Linear(d_hid, 2)
        self.proj = nn.Linear(d_hid, d_proj)
    def forward(self, x, return_rep=False):
        e = self.emb(x)
        _, h = self.gru(e)
        h_last = h[-1]
        logits = self.classifier(h_last)
        if return_rep:
            z_lang = self.proj(h_last)  # (B, D_PROJ)
            return logits, z_lang
        return logits

def make_model(n_tokens=VOCAB_SIZE):
    benc = AdaptiveBrainEncoder(d_proj=D_PROJ)
    sgru = TaskModelGRU(n_tokens=n_tokens, d_proj=D_PROJ)
    return benc, sgru

print('Unified models with aligned projection dimension (D_PROJ=64) initialized.')


Unified models with aligned projection dimension (D_PROJ=64) initialized.


In [4]:
X8, M8 = None, None
if P8_X_PATH and P8_X_PATH.exists():
    X8 = np.load(P8_X_PATH)
    M8 = pd.read_csv(P8_M_PATH) if P8_M_PATH else None
    print('Phase 8 X shape:', X8.shape)
    if M8 is not None:
        print('Phase 8 M shape:', M8.shape)
else:
    print('Notice: Phase 8 reference data not detected.')


Phase 8 X shape: (3353, 4, 16, 5)
Phase 8 M shape: (3353, 15)


In [5]:
X9, M9 = None, None
if P9A_X_PATH and P9A_X_PATH.exists() and P9A_M_PATH and P9A_M_PATH.exists():
    X9 = np.load(P9A_X_PATH)
    M9 = pd.read_csv(P9A_M_PATH)
    print('Phase 9a X shape:', X9.shape)
    print('Phase 9a M shape:', M9.shape)
    if 'recalled' in M9.columns:
        print('Phase 9a mean recall:', round(float(M9['recalled'].mean()), 3))
else:
    print('Phase 9a dataset not detected.')


Phase 9a X shape: (6122, 4, 5)
Phase 9a M shape: (6122, 6)
Phase 9a mean recall: 0.244


In [6]:
def make_maxent(X):
    """Maximum-entropy control: shuffle across trials per feature independently."""
    rng = np.random.default_rng(42)
    X_me = X.copy()
    if X_me.ndim == 3:
        n, r, b = X_me.shape
        for ri in range(r):
            for bi in range(b):
                X_me[:, ri, bi] = rng.permutation(X_me[:, ri, bi])
    elif X_me.ndim == 4:
        n, r, t, b = X_me.shape
        for ri in range(r):
            for ti in range(t):
                for bi in range(b):
                    X_me[:, ri, ti, bi] = rng.permutation(X_me[:, ri, ti, bi])
    return X_me

def make_phase_scrambled(X):
    """Phase-scramble control: randomize phase in frequency domain."""
    rng = np.random.default_rng(123)
    X_ps = np.zeros_like(X)
    if X_ps.ndim == 3:
        n, r, b = X.shape
        for ri in range(r):
            for bi in range(b):
                col = X[:, ri, bi]
                f   = np.fft.rfft(col)
                ph  = rng.uniform(0, 2 * np.pi, size=len(f))
                f2  = np.abs(f) * np.exp(1j * ph)
                X_ps[:, ri, bi] = np.fft.irfft(f2, n=n)
    elif X_ps.ndim == 4:
        n, r, t, b = X.shape
        for ri in range(r):
            for ti in range(t):
                for bi in range(b):
                    col = X[:, ri, ti, bi]
                    f   = np.fft.rfft(col)
                    ph  = rng.uniform(0, 2 * np.pi, size=len(f))
                    f2  = np.abs(f) * np.exp(1j * ph)
                    X_ps[:, ri, ti, bi] = np.fft.irfft(f2, n=n)
    return X_ps

if X9 is not None:
    X9_me = make_maxent(X9)
    X9_ps = make_phase_scrambled(X9)
    print('Controls generated for ds004789:')
    print('  MaxEnt shape        :', X9_me.shape)
    print('  Phase-scramble shape:', X9_ps.shape)
else:
    X9_me = X9_ps = None
    print('Skipping controls (Phase 9a not loaded).')


Controls generated for ds004789:
  MaxEnt shape        : (6122, 4, 5)
  Phase-scramble shape: (6122, 4, 5)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def loso_decode(X, y, n_splits=5):
    X_flat = X.reshape(len(X), -1)
    y = np.asarray(y).astype(int)
    valid = ~np.isnan(X_flat).any(axis=1) & ~np.isnan(y)
    X_flat, y = X_flat[valid], y[valid]
    if len(np.unique(y)) < 2:
        return float('nan')
    n = len(X_flat)
    fold_size = max(1, n // n_splits)
    accs = []
    for i in range(n_splits):
        test_idx  = np.arange(i * fold_size, min((i + 1) * fold_size, n))
        train_idx = np.setdiff1d(np.arange(n), test_idx)
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue
        y_train = y[train_idx]
        if len(np.unique(y_train)) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200, C=1.0))
        clf.fit(X_flat[train_idx], y_train)
        pred = clf.predict(X_flat[test_idx])
        classes = np.unique(y[test_idx])
        per_class = [np.mean(pred[y[test_idx]==c] == c) for c in classes]
        accs.append(np.mean(per_class))
    return float(np.mean(accs)) if accs else float('nan')

if X9 is not None and 'recalled' in M9.columns:
    y9 = M9['recalled'].fillna(0).astype(int).values
    gate_acc = loso_decode(X9, y9)
    print(f'ds004789 Gate Decoding Accuracy (Memory Recall): {gate_acc:.3f}')
    print('Gate Status:', 'PASSED' if gate_acc > 0.52 else 'MARGINAL')
else:
    print('Gate test skipped.')


ds004789 Gate Decoding Accuracy (Memory Recall): 0.500
Gate Status: MARGINAL


In [8]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Compute device:', DEVICE)

def infonce_loss(z_brain, z_lang, temperature=0.1):
    z_b = F.normalize(z_brain, dim=-1)
    z_l = F.normalize(z_lang,  dim=-1)
    logits = z_b @ z_l.T / temperature
    labels = torch.arange(len(z_b), device=logits.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

def run_b2m(X_np, y_np, seqs_np, n_epochs=40, lr=3e-4, batch=64, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_np    = np.nan_to_num(X_np, nan=0.0)
    y_np    = np.nan_to_num(y_np, nan=0.0).astype(int)
    seqs_np = np.nan_to_num(seqs_np, nan=0).astype(int)

    n = len(X_np)
    idx = np.random.permutation(n)
    split = int(0.8 * n)
    tr_idx, va_idx = idx[:split], idx[split:]

    X_tr  = torch.tensor(X_np[tr_idx],   dtype=torch.float32, device=DEVICE)
    y_tr  = torch.tensor(y_np[tr_idx],   dtype=torch.long,    device=DEVICE)
    sq_tr = torch.tensor(seqs_np[tr_idx], dtype=torch.long,    device=DEVICE)
    X_va  = torch.tensor(X_np[va_idx],   dtype=torch.float32, device=DEVICE)
    y_va  = torch.tensor(y_np[va_idx],   dtype=torch.long,    device=DEVICE)
    sq_va = torch.tensor(seqs_np[va_idx], dtype=torch.long,    device=DEVICE)

    benc, sgru = make_model(n_tokens=VOCAB_SIZE)
    benc = benc.to(DEVICE)
    sgru = sgru.to(DEVICE)
    opt  = torch.optim.Adam(list(benc.parameters()) + list(sgru.parameters()), lr=lr)

    for ep in range(n_epochs):
        benc.train(); sgru.train()
        perm = torch.randperm(len(X_tr), device=DEVICE)
        for i in range(0, len(X_tr), batch):
            bi  = perm[i:i+batch]
            xb  = X_tr[bi]; yb = y_tr[bi]; sqb = sq_tr[bi]
            z_b = benc(xb)
            logits, z_l = sgru(sqb, return_rep=True)
            loss = F.cross_entropy(logits, yb) + 0.5 * infonce_loss(z_b, z_l)
            opt.zero_grad()
            loss.backward()
            opt.step()

    benc.eval(); sgru.eval()
    with torch.no_grad():
        tr_acc = float((sgru(sq_tr).argmax(1) == y_tr).float().mean())
        va_acc = float((sgru(sq_va).argmax(1) == y_va).float().mean())
        z_b_all = benc(X_va).cpu().numpy()
        _, z_l_all = sgru(sq_va, return_rep=True)
        z_l_all = z_l_all.cpu().numpy()

        def lin_cka(A, B):
            A = A - A.mean(0); B = B - B.mean(0)
            num = np.linalg.norm(A.T @ B, 'fro') ** 2
            den = np.linalg.norm(A.T @ A, 'fro') * np.linalg.norm(B.T @ B, 'fro')
            return float(num / (den + 1e-12))
        cka = lin_cka(z_b_all, z_l_all)
    return {'train_acc': tr_acc, 'val_acc': va_acc, 'cka': cka}

print('B2M training engine defined.')


Compute device: cuda
B2M training engine defined.


In [9]:
def build_sequences(M, max_len=12, pad=0, sep=1):
    seqs, labels = [], []
    rec_col = 'recalled' if 'recalled' in M.columns else None
    spos_col = 'serialpos' if 'serialpos' in M.columns else None

    for _, row in M.iterrows():
        rec = int(row.get(rec_col, 0) or 0) if rec_col else 0
        spos = int(row.get(spos_col, 1) or 1) if spos_col else 1
        tok = min(max(spos, 1) + 1, max_len + 1)
        seq = [sep, tok] + [pad] * max_len
        seqs.append(seq[:max_len + 2])
        labels.append(rec)
    return np.array(seqs, dtype=int), np.array(labels, dtype=int)

seqs9, y9 = (build_sequences(M9) if M9 is not None else (None, None))
seqs8, y8 = (build_sequences(M8) if M8 is not None else (None, None))

if seqs9 is not None:
    print('ds004789 sequences:', seqs9.shape, 'labels:', y9.shape)
if seqs8 is not None:
    print('Phase 8 sequences :', seqs8.shape, 'labels:', y8.shape)


ds004789 sequences: (6122, 14) labels: (6122,)
Phase 8 sequences : (3353, 14) labels: (3353,)


In [10]:
N_SEEDS = 5
results = []
datasets = []

if X8 is not None and seqs8 is not None:
    datasets.append(('Phase 8 Reference', X8, y8, seqs8))
if X9 is not None and seqs9 is not None:
    datasets.append(('ds004789 Real Brain', X9, y9, seqs9))
    datasets.append(('ds004789 MaxEnt', X9_me, y9, seqs9))
    datasets.append(('ds004789 Phase Scrambled', X9_ps, y9, seqs9))

for ds_name, X_ds, y_ds, seqs_ds in datasets:
    print(f'\nRunning evaluation for: {ds_name}')
    for seed in range(N_SEEDS):
        res = run_b2m(X_ds, y_ds, seqs_ds, seed=seed)
        res['condition'] = ds_name
        res['seed'] = seed
        results.append(res)
        print(f'  Seed {seed} | Val Acc: {res["val_acc"]:.3f} | CKA: {res["cka"]:.3f}')

res_df = pd.DataFrame(results)
print('\n=== Condition Summary ===')
print(res_df.groupby('condition')[['val_acc', 'cka']].mean().round(3).to_string())



Running evaluation for: Phase 8 Reference
  Seed 0 | Val Acc: 1.000 | CKA: 0.000
  Seed 1 | Val Acc: 1.000 | CKA: 0.000
  Seed 2 | Val Acc: 1.000 | CKA: 0.000
  Seed 3 | Val Acc: 1.000 | CKA: 0.000
  Seed 4 | Val Acc: 1.000 | CKA: 0.000

Running evaluation for: ds004789 Real Brain
  Seed 0 | Val Acc: 0.758 | CKA: 0.004
  Seed 1 | Val Acc: 0.768 | CKA: 0.005
  Seed 2 | Val Acc: 0.762 | CKA: 0.005
  Seed 3 | Val Acc: 0.770 | CKA: 0.004
  Seed 4 | Val Acc: 0.741 | CKA: 0.006

Running evaluation for: ds004789 MaxEnt
  Seed 0 | Val Acc: 0.762 | CKA: 0.005
  Seed 1 | Val Acc: 0.768 | CKA: 0.005
  Seed 2 | Val Acc: 0.762 | CKA: 0.007
  Seed 3 | Val Acc: 0.770 | CKA: 0.004
  Seed 4 | Val Acc: 0.741 | CKA: 0.004

Running evaluation for: ds004789 Phase Scrambled
  Seed 0 | Val Acc: 0.758 | CKA: 0.008
  Seed 1 | Val Acc: 0.768 | CKA: 0.006
  Seed 2 | Val Acc: 0.762 | CKA: 0.008
  Seed 3 | Val Acc: 0.770 | CKA: 0.008
  Seed 4 | Val Acc: 0.741 | CKA: 0.006

=== Condition Summary ===
              

In [11]:
stat_rows = []

if X9 is not None:
    real_acc = res_df[res_df['condition'] == 'ds004789 Real Brain']['val_acc'].values
    me_acc   = res_df[res_df['condition'] == 'ds004789 MaxEnt']['val_acc'].values
    ps_acc   = res_df[res_df['condition'] == 'ds004789 Phase Scrambled']['val_acc'].values

    delta_me = float(np.mean(real_acc) - np.mean(me_acc))
    t_me, p_me = stats.ttest_ind(real_acc, me_acc)
    stat_rows.append({
        'Comparison': 'Real Brain vs MaxEnt (ds004789)',
        'Delta_Causal': round(delta_me, 4),
        't_stat': round(float(t_me), 3),
        'p_val': round(float(p_me), 4),
    })

    delta_ps = float(np.mean(real_acc) - np.mean(ps_acc))
    t_ps, p_ps = stats.ttest_ind(real_acc, ps_acc)
    stat_rows.append({
        'Comparison': 'Real Brain vs Phase Scramble (ds004789)',
        'Delta_Causal': round(delta_ps, 4),
        't_stat': round(float(t_ps), 3),
        'p_val': round(float(p_ps), 4),
    })

stat_df = pd.DataFrame(stat_rows)
print(stat_df.to_string(index=False))
stat_df.to_csv(OUT_DIR / 'replication_stats.csv', index=False)


                             Comparison  Delta_Causal  t_stat  p_val
        Real Brain vs MaxEnt (ds004789)       -0.0008  -0.113  0.913
Real Brain vs Phase Scramble (ds004789)        0.0000   0.000  1.000


In [12]:
summary_rows = []
for c in res_df['condition'].unique():
    sub = res_df[res_df['condition'] == c]
    summary_rows.append({
        'Condition': c,
        'Seeds': len(sub),
        'Val_Accuracy_Mean': round(float(sub['val_acc'].mean()), 3),
        'Val_Accuracy_Std':  round(float(sub['val_acc'].std()), 3),
        'CKA_Mean':          round(float(sub['cka'].mean()), 3),
        'CKA_Std':           round(float(sub['cka'].std()), 3),
    })
rep_table = pd.DataFrame(summary_rows)
print(rep_table.to_string(index=False))
rep_table.to_csv(OUT_DIR / 'replication_table.csv', index=False)
print('\nSaved replication_table.csv')


               Condition  Seeds  Val_Accuracy_Mean  Val_Accuracy_Std  CKA_Mean  CKA_Std
       Phase 8 Reference      5              1.000             0.000     0.000    0.000
     ds004789 Real Brain      5              0.760             0.011     0.005    0.001
         ds004789 MaxEnt      5              0.761             0.011     0.005    0.001
ds004789 Phase Scrambled      5              0.760             0.011     0.007    0.001

Saved replication_table.csv


In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot of Accuracy
ax1 = axes[0]
conds = res_df['condition'].unique()
means = [res_df[res_df['condition']==c]['val_acc'].mean() for c in conds]
sems  = [res_df[res_df['condition']==c]['val_acc'].sem() for c in conds]
ax1.bar(range(len(conds)), means, yerr=sems, color=['#2b5c8f', '#2ca02c', '#d62728', '#9467bd'][:len(conds)], capsize=5)
ax1.set_xticks(range(len(conds)))
ax1.set_xticklabels([c.replace(' ', '\n') for c in conds], rotation=15)
ax1.set_ylabel('Validation Accuracy')
ax1.set_title('Replication Performance Across Conditions')
ax1.axhline(0.5, ls='--', color='gray', alpha=0.7, label='Chance')
ax1.legend()

# CKA vs Accuracy scatter
ax2 = axes[1]
for c in conds:
    sub = res_df[res_df['condition']==c]
    ax2.scatter(sub['cka'], sub['val_acc'], label=c, s=50, edgecolors='k')
ax2.set_xlabel('CKA (Brain-Language Representational Similarity)')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('CKA vs Downstream Performance')
ax2.legend()

plt.tight_layout()
fig_path = OUT_DIR / 'replication_fig.png'
plt.savefig(fig_path, dpi=200)
plt.close()
print('Saved publication figure to:', fig_path)


Saved publication figure to: /kaggle/working/phase9b_out/replication_fig.png


In [14]:
print('=' * 60)
print('PHASE 9b FINAL REPLICATION VERDICT')
print('=' * 60)

if X9 is not None:
    acc_real = float(res_df[res_df['condition']=='ds004789 Real Brain']['val_acc'].mean())
    acc_me   = float(res_df[res_df['condition']=='ds004789 MaxEnt']['val_acc'].mean())
    delta = acc_real - acc_me
    verdict = 'SUCCESSFULLY REPLICATED' if delta > 0 else 'NOT REPLICATED'
    print(f'Real Brain Accuracy : {acc_real:.3f}')
    print(f'MaxEnt Accuracy     : {acc_me:.3f}')
    print(f'Delta Advantage     : {delta:+.3f}')
    print(f'Final Verdict       : {verdict}')
else:
    print('Replication data not attached.')

res_df.to_csv(OUT_DIR / 'results_all.csv', index=False)
print('\nAll outputs stored in:', OUT_DIR)


PHASE 9b FINAL REPLICATION VERDICT
Real Brain Accuracy : 0.760
MaxEnt Accuracy     : 0.761
Delta Advantage     : -0.001
Final Verdict       : NOT REPLICATED

All outputs stored in: /kaggle/working/phase9b_out
